In [1]:
import os

from agent_framework import Agent, MCPStdioTool, MCPStreamableHTTPTool
from dotenv import load_dotenv

from azure_client import create_chat_client

load_dotenv()

True

In [2]:
tool = MCPStreamableHTTPTool(
    name="Microsoft Learn MCP",
    url=os.getenv("MCP_LEARN_URL", "https://learn.microsoft.com/api/mcp"),
    # we don't require approval for microsoft_docs_search tool calls
    # but we do for any other tool
    # approval_mode={"never_require_approval": ["microsoft_docs_search"]},
)

In [3]:
# Optional: requires npx on PATH
tool2 = MCPStdioTool(
    name="filesystem",
    command="npx",
    args=[
        "-y",
        "@modelcontextprotocol/server-filesystem",
        os.path.expanduser("~/Documents"),
    ],
    description="File system operations",
)

In [4]:
tool3 = MCPStreamableHTTPTool(
    name="localhost MCP",
    url="http://localhost:1337/mcp",
)

In [5]:
import json
import logging

import httpx
from httpx._content import ByteStream

original_request = None
# Restore any previous request patch from earlier runs
if "original_request" in globals():
    httpx.AsyncClient.request = original_request
    del globals()["original_request"]

# Reset root handlers and silence everything except our custom httpx logs
root_logger = logging.getLogger()
for handler in list(root_logger.handlers):
    root_logger.removeHandler(handler)
logging.basicConfig(level=logging.ERROR)


# Limit output to debug entries from httpx.logged_send_single_request
class _HttpxFunctionFilter(logging.Filter):
    def filter(self, record: logging.LogRecord) -> bool:
        return record.funcName == "logged_send_single_request"


httpx_logger = logging.getLogger("httpx")
httpx_logger.handlers.clear()
httpx_logger.setLevel(logging.DEBUG)
httpx_logger.propagate = False

httpx_handler = logging.StreamHandler()
httpx_handler.setLevel(logging.DEBUG)
httpx_handler.setFormatter(logging.Formatter("\n[%(name)s - %(module)s] %(message)s"))
# httpx_handler.addFilter(_HttpxFunctionFilter())
httpx_logger.addHandler(httpx_handler)


# Also enable openai library logging if available
# logging.getLogger("openai").setLevel(logging.DEBUG)

if not hasattr(httpx.AsyncClient, "_original_send_single_request"):
    httpx.AsyncClient._original_send_single_request = (
        httpx.AsyncClient._send_single_request
    )


def pretty_print_json_in_text(text: str) -> str:
    """Try to find and pretty-print JSON in text, including SSE format."""
    lines = text.split("\n")
    result_lines = []

    for line in lines:
        # Check if this is an SSE data line
        if line.startswith("data: "):
            json_str = line[6:]  # Remove 'data: ' prefix
            try:
                json_obj = json.loads(json_str)
                pretty_json = json.dumps(json_obj, indent=2)
                # Indent each line of the pretty JSON for better formatting
                indented_json = "\n".join(
                    "    " + line for line in pretty_json.split("\n")
                )
                result_lines.append(f"data: \n{indented_json}")
            except json.JSONDecodeError:
                result_lines.append(line)
        else:
            result_lines.append(line)

    return "\n".join(result_lines)


async def logged_send_single_request(self, request):
    logger = logging.getLogger("httpx")
    if logger.isEnabledFor(logging.DEBUG):
        stream = request.stream
        if isinstance(stream, ByteStream):
            body = stream._stream
            logger.debug("Request: %s %s", request.method, request.url)
            if body:
                try:
                    body_str = body.decode("utf-8")
                    # Try to parse and pretty-print JSON
                    try:
                        json_obj = json.loads(body_str)
                        pretty_json = json.dumps(json_obj, indent=2)
                        logger.debug("\tRequest body (JSON):\n%s", pretty_json)
                    except json.JSONDecodeError:
                        logger.debug("\tRequest body (utf-8): %s", body_str)
                except UnicodeDecodeError:
                    logger.debug("\tRequest body (bytes): %s", body)
            else:
                logger.debug("\tRequest body: <empty>")
        else:
            logger.debug(
                "\tRequest body not logged (stream type: %s)", type(stream).__name__
            )
    response = await httpx.AsyncClient._original_send_single_request(self, request)
    if logger.isEnabledFor(logging.DEBUG):
        # logger.debug("Response headers: %s", dict(response.headers))
        response_body = await response.aread()
        try:
            response_str = response_body.decode("utf-8")
            # Try to parse and pretty-print JSON (handles both pure JSON and SSE format)
            try:
                json_obj = json.loads(response_str)
                pretty_json = json.dumps(json_obj, indent=2)
                logger.debug(
                    f"\tResponse {response.status_code} (JSON):\n{pretty_json}"
                )
            except json.JSONDecodeError:
                # Might be SSE format or other text
                pretty_text = pretty_print_json_in_text(response_str)
                logger.debug(f"\tResponse {response.status_code}:\n{pretty_text}")
        except UnicodeDecodeError:
            logger.debug(f"\tResponse {response.status_code} (bytes): {response_body}")
    return response


if not getattr(httpx.AsyncClient, "_logging_patch_applied", False):
    httpx.AsyncClient._send_single_request = logged_send_single_request
    httpx.AsyncClient._logging_patch_applied = True

In [6]:
# Entra ID auth against Azure Gov; see azure_client.py
llm = create_chat_client()

agent = Agent(
    llm,
    "You are a helpful agent. You use Model Context Protocol (MCP) tools to answer user questions. "
    "You can only respond using the tools available to you. Do not make up tool functionality. The tools will be"
    "Provided to you in the prompt.",
    name="test_agent",
    tools=[tool],
)

query = "tell me about Azure OpenAI?"
print(f"User: {query}")
print("\n=== HTTP Request/Response Details Below ===\n")
result = await agent.run(query)
print("\n=== End of HTTP Details ===\n")
print(f"Agent: {result}\n")

User: tell me about Azure OpenAI?

=== HTTP Request/Response Details Below ===




[httpx - 560776882] Request: POST https://learn.microsoft.com/api/mcp



[httpx - 560776882] 	Request body (JSON):
{
  "method": "initialize",
  "params": {
    "protocolVersion": "2025-11-25",
    "capabilities": {
      "sampling": {}
    },
    "clientInfo": {
      "name": "mcp",
      "version": "0.1.0"
    }
  },
  "jsonrpc": "2.0",
  "id": 0
}



[httpx - _client] HTTP Request: POST https://learn.microsoft.com/api/mcp "HTTP/1.1 200 OK"



[httpx - 560776882] 	Response 200:
event: message
data: 
    {
      "result": {
        "protocolVersion": "2025-06-18",
        "capabilities": {
          "logging": {},
          "prompts": {
            "listChanged": true
          },
          "resources": {
            "listChanged": true
          },
          "tools": {
            "listChanged": true
          }
        },
        "serverInfo": {
          "name": "Microsoft Learn MCP Server",
          "version": "1.0.0"
        },
        "instructions": "# Microsoft Learn MCP Server\r\n\r\nThis server gives structured access to official Microsoft and Azure documentation via three tools:\r\n\r\n## Tools\r\n\r\n### microsoft_docs_search\r\nSearch official documentation and return up to 10 concise, high-quality content chunks (max 500 tokens each), including title, URL, and excerpt.\r\n\r\n- Use first to get a quick, reliable overview\r\n- Ideal for grounding answers in Microsoft knowledge\r\n\r\n### microsoft_code_sample_s

...


[httpx - 560776882] Request: POST https://azd-mcp-client-openai-awevs6sfsjhxq.openai.azure.us/openai/v1/responses?api-version=preview



[httpx - 560776882] 	Request body (JSON):
{
  "input": [
    {
      "call_id": "call_1Vyp9xPWZ0WpyVehpdJkrdkR",
      "type": "function_call_output",
      "output": "{\"results\":[{\"title\":\"What is Azure OpenAI in Azure AI Foundry Models?\",\"content\":\"# What is Azure OpenAI in Azure AI Foundry Models?\\n## Get started with Azure OpenAI\\nTo get started with Azure OpenAI, you need to create an Azure OpenAI resource in your Azure subscription.\\nStart with the [Create and deploy an Azure OpenAI resource](https://learn.microsoft.com/azure/ai-foundry/openai/how-to/create-resource) guide.\\n1. You can create a resource via Azure portal, Azure CLI, or Azure PowerShell.\\n2. When you have an Azure OpenAI resource, you can deploy a model such as GPT-4o.\\n3. When you have a deployed model, you can:\\n3.1. Try out the [Azure AI Foundry portal](https://ai.azure.com/?cid=learnDocs) playgrounds to explore the capabilities of the models.\\n3.2. You can also just start making API calls to t


[httpx - _client] HTTP Request: POST https://azd-mcp-client-openai-awevs6sfsjhxq.openai.azure.us/openai/v1/responses?api-version=preview "HTTP/1.1 200 OK"



[httpx - 560776882] 	Response 200 (JSON):
{
  "id": "resp_053a0e7d960d6a32006a8cb42baf408193b9cbb34785d037b5",
  "object": "response",
  "created_at": 1787606059,
  "status": "completed",
  "background": false,
  "completed_at": 1787606070,
  "content_filters": [
    {
      "blocked": false,
      "source_type": "completion",
      "content_filter_raw": [],
      "content_filter_results": {
        "hate": {
          "filtered": false,
          "severity": "safe"
        },
        "sexual": {
          "filtered": false,
          "severity": "safe"
        },
        "violence": {
          "filtered": false,
          "severity": "safe"
        },
        "self_harm": {
          "filtered": false,
          "severity": "safe"
        }
      },
      "content_filter_offsets": {
        "start_offset": 0,
        "end_offset": 3173,
        "check_offset": 0
      }
    }
  ],
  "error": null,
  "frequency_penalty": 0.0,
  "incomplete_details": null,
  "instructions": null,
  "m


=== End of HTTP Details ===

Agent: Azure OpenAI is a Microsoft Azure service that gives you API access to OpenAI’s models (like GPT‑4o, GPT‑4.1, GPT‑3.5, embeddings, image, and some reasoning models) with Azure’s enterprise security, compliance, and management features.

Here’s the core idea and what you can do with it:

1. **What it is**
   - A managed Azure service that exposes OpenAI models via:
     - REST APIs
     - SDKs (Python, C#, JavaScript/TypeScript, Java, Go, etc.)
   - It runs the *same* model families you get from OpenAI (GPT‑4o, GPT‑4.1, GPT‑3.5‑Turbo, embeddings, DALL·E/image models, etc.), but hosted in Azure regions.

2. **Key capabilities**
   - **Text & chat**: content generation, summarization, Q&A, translation, classification, natural language to SQL/code, etc.
   - **Vision & multimodal**: GPT‑4o / GPT‑4 Turbo with Vision for understanding images and mixed text+image prompts.
   - **Code**: generate and refactor code, explain code, build copilots for developer

In [7]:
query = "greet me?"
print(f"User: {query}")
print("\n=== HTTP Request/Response Details Below ===\n")
result = await agent.run(query)
print("\n=== End of HTTP Details ===\n")
print(f"Agent: {result}\n")


[httpx - 560776882] Request: POST https://azd-mcp-client-openai-awevs6sfsjhxq.openai.azure.us/openai/v1/responses?api-version=preview



[httpx - 560776882] 	Request body (JSON):
{
  "include": [
    "reasoning.encrypted_content"
  ],
  "input": [
    {
      "type": "message",
      "role": "system",
      "content": [
        {
          "type": "input_text",
          "text": "You are a helpful agent. You use Model Context Protocol (MCP) tools to answer user questions. You can only respond using the tools available to you. Do not make up tool functionality. The tools will beProvided to you in the prompt."
        }
      ]
    },
    {
      "type": "message",
      "role": "user",
      "content": [
        {
          "type": "input_text",
          "text": "greet me?"
        }
      ]
    }
  ],
  "model": "gpt-5.1",
  "stream": false,
  "tool_choice": "auto",
  "tools": [
    {
      "name": "microsoft_docs_search",
      "parameters": {
        "type": "object",
        "properties": {
          "query": {
            "description": "a query or topic about Microsoft/Azure products, services, platforms, develop

User: greet me?

=== HTTP Request/Response Details Below ===




[httpx - _client] HTTP Request: POST https://azd-mcp-client-openai-awevs6sfsjhxq.openai.azure.us/openai/v1/responses?api-version=preview "HTTP/1.1 200 OK"



[httpx - 560776882] 	Response 200 (JSON):
{
  "id": "resp_0282ee8bef345a92006a8cb4373270819687b81894b6428fcd",
  "object": "response",
  "created_at": 1787606071,
  "status": "completed",
  "background": false,
  "completed_at": 1787606072,
  "content_filters": [
    {
      "blocked": false,
      "source_type": "prompt",
      "content_filter_raw": [],
      "content_filter_results": {
        "hate": {
          "filtered": false,
          "severity": "safe"
        },
        "sexual": {
          "filtered": false,
          "severity": "safe"
        },
        "violence": {
          "filtered": false,
          "severity": "safe"
        },
        "self_harm": {
          "filtered": false,
          "severity": "safe"
        }
      },
      "content_filter_offsets": {
        "start_offset": 0,
        "end_offset": 42,
        "check_offset": 0
      }
    },
    {
      "blocked": false,
      "source_type": "completion",
      "content_filter_raw": [],
      "content_f


=== End of HTTP Details ===

Agent: Hello! Nice to meet you. How’s your day going so far?



In [8]:
from random import randint
from typing import Annotated


def get_weather(
    location: Annotated[str, "The location to get the weather for."],
) -> str:
    """Get the weather for a given location."""
    conditions = ["sunny", "cloudy", "rainy", "stormy"]
    return f"The weather in {location} is {conditions[randint(0, 3)]} with a high of {randint(10, 30)}°C."


weather_agent = Agent(
    llm,
    "You are a helpful weather agent.",
    name="WeatherAgent",
    tools=[get_weather],
)


async def non_streaming_example() -> None:
    """Example of non-streaming response (get the complete result at once)."""
    print("=== Non-streaming Response Example ===")

    query = "What's the weather like in Seattle?"
    print(f"User: {query}")
    result = await weather_agent.run(query)
    print(f"Agent: {result}\n")


async def streaming_example() -> None:
    """Example of streaming response (get results as they are generated)."""
    print("=== Streaming Response Example ===")

    query = "What's the weather like in Portland?"
    print(f"User: {query}")
    print("Agent: ", end="", flush=True)
    async for chunk in weather_agent.run(query, stream=True):
        if chunk.text:
            print(chunk.text, end="", flush=True)
    print("\n")

In [9]:
await streaming_example()

=== Streaming Response Example ===
User: What's the weather like in Portland?
Agent: 


[httpx - 560776882] Request: POST https://azd-mcp-client-openai-awevs6sfsjhxq.openai.azure.us/openai/v1/responses?api-version=preview



[httpx - 560776882] 	Request body (JSON):
{
  "include": [
    "reasoning.encrypted_content"
  ],
  "input": [
    {
      "type": "message",
      "role": "system",
      "content": [
        {
          "type": "input_text",
          "text": "You are a helpful weather agent."
        }
      ]
    },
    {
      "type": "message",
      "role": "user",
      "content": [
        {
          "type": "input_text",
          "text": "What's the weather like in Portland?"
        }
      ]
    }
  ],
  "model": "gpt-5.1",
  "stream": true,
  "tool_choice": "auto",
  "tools": [
    {
      "name": "get_weather",
      "parameters": {
        "properties": {
          "location": {
            "description": "The location to get the weather for.",
            "title": "Location",
            "type": "string"
          }
        },
        "required": [
          "location"
        ],
        "title": "get_weather_input",
        "type": "object",
        "additionalProperties": false
   


[httpx - _client] HTTP Request: POST https://azd-mcp-client-openai-awevs6sfsjhxq.openai.azure.us/openai/v1/responses?api-version=preview "HTTP/1.1 200 OK"



[httpx - 560776882] 	Response 200:
event: response.created
data: 
    {
      "type": "response.created",
      "response": {
        "id": "resp_0460c4f38417ece0006a8cb43870108193ac79891e984d1b84",
        "object": "response",
        "created_at": 1787606072,
        "status": "in_progress",
        "background": false,
        "completed_at": null,
        "content_filters": null,
        "error": null,
        "frequency_penalty": 0.0,
        "incomplete_details": null,
        "instructions": null,
        "max_output_tokens": null,
        "max_tool_calls": null,
        "model": "gpt-5.1",
        "moderation": null,
        "output": [],
        "parallel_tool_calls": true,
        "presence_penalty": 0.0,
        "previous_response_id": null,
        "prompt_cache_key": null,
        "prompt_cache_retention": "in_memory",
        "reasoning": {
          "context": "current_turn",
          "effort": "none",
          "mode": "standard",
          "summary": null
        },

...

 high

 of

10

°C

.

In [10]:
# MCP transports must be closed in the task that opened them. Skipping this leaves the
# connection to be finalized by the garbage collector, which raises
# "Attempted to exit cancel scope in a different task than it was entered in".
for mcp_tool in [tool, tool3]:
    if mcp_tool.is_connected:
        await mcp_tool.close()
        print(f"closed {mcp_tool.name}")


closed Microsoft Learn MCP
